# 02 · 预测基线与自定义预测器对比

基于 `src/` 下三个模块：`data_loader` → `baseline_forecasters` → `custom_forecaster`。

流程：加载数据（**合成 AR(1) 数据**，seed=42 可复现，参数对齐 config 基准 4 SKU × 365 天；
M5 加载路径已实现、本次未启用）→
**长表**（date / store_id / item_id / demand，团队注意事项 §8 口径）→
需求交接对象（ReplayDemandGenerator）验证 → 三种官方基线滚动预测误差对比 →
加入自定义 `WeightedMAForecaster` → 可视化预测均值与 std 不确定区间 →
误差指标落盘 `outputs/results/`、图落盘 `outputs/figures/`。

> 本 notebook 只做**数据与预测**：不读取 `config/network_config.yaml` 的内容，
> 不构建供应链网络、不做补货仿真（属拓扑/策略模块）。
> 绘图逻辑沉淀在 `src/visualization/dashboard.py`（预测绘图函数）。

In [ ]:
# ---- 环境导入：兼容内核工作目录在项目根或 notebooks/ 两种情况 ----
import sys
from pathlib import Path

_root = Path.cwd()
if _root.name == "notebooks":
    _root = _root.parent
sys.path.insert(0, str(_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import (
    load_m5_demand, load_synthetic_demand, build_replay_generator, pivot_demand_wide)
from src.forecasting.baseline_forecasters import (
    run_baseline_comparison, generate_baseline_forecasts, save_baseline_metrics)
from src.forecasting.custom_forecaster import WeightedMAForecaster, run_comparison_with_custom
from src.visualization.dashboard import (
    plot_demand_overview, plot_forecast_mean, plot_uncertainty_band)
from deepbullwhip.demand.replay import ReplayDemandGenerator

# 输出目录：图 -> outputs/figures/，误差指标 -> outputs/results/（save_baseline_metrics 内部自建）
FIGURES_DIR = _root / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"项目根: {_root}")
print(f"图片目录: {FIGURES_DIR}")

In [ ]:
# ---- 加载数据 ------------------------------------------------------------
# 【合成数据】演示默认：框架 AR(1)，默认参数对齐 config/network_config.yaml 基准
#   （sku_count=5、simulation_days=365、demand.mean=10、demand.std=3，seed 可复现）。
# 【M5 真实数据】需要 data/raw/m5/sales_train_evaluation.csv（本地放置，不联网），
#   weekly 口径（周初日期）；两套口径同为长表 date/store_id/item_id/demand。
USE_M5 = False   # <-- 拿到 M5 本地 CSV 后改为 True 即切换真实数据
STORE, DEPTS = "CA_1", ["FOODS_1", "FOODS_2", "HOUSEHOLD_1", "HOUSEHOLD_2"]

if USE_M5:
    try:
        demand_df = load_m5_demand(STORE, DEPTS)
        data_source = f"M5 {STORE} x{len(DEPTS)}"        # 英文安全标签，用于图标题
        print(f"[M5] 加载成功：{len(DEPTS)} 条 dept 级序列（weekly，周初日期）")
    except (FileNotFoundError, ImportError, ValueError) as exc:
        # M5 不可用时不静默：打印原因后降级合成，notebook 依旧可跑通
        print(f"[M5] 加载失败（{type(exc).__name__}），自动降级为合成数据：\n  {exc}")
        demand_df = load_synthetic_demand()
        data_source = "synthetic AR(1) (M5 fallback)"
else:
    # ---- 合成数据（演示默认，config 对齐） ----
    demand_df = load_synthetic_demand()
    data_source = "synthetic AR(1)"
    print("[合成] 使用框架 AR(1) 合成需求（对齐 config：5 SKU x 365 天、mean=10、std=3）")

print(f"数据来源: {data_source} | 长表形状 (行, 列): {demand_df.shape}")
print(f"期数（unique date）: {demand_df['date'].nunique()} | "
      f"序列数（unique item_id）: {demand_df['item_id'].nunique()}")

# ---- 需求交接对象验证：build_replay_generator ----
# 整表（长表）→ 按 date 聚合求和成单序列；逐节点 → 按 item_id 过滤后各自构造
replay_gen = build_replay_generator(demand_df)
assert isinstance(replay_gen, ReplayDemandGenerator)
assert replay_gen.generate(T=600).shape == (600,)         # 超长自动循环填充
node_generators = {f"{s}_{i}": build_replay_generator(g)
                   for (s, i), g in demand_df.groupby(["store_id", "item_id"],
                                                      sort=True)}
assert all(isinstance(g, ReplayDemandGenerator) for g in node_generators.values())
print(f"交接对象 OK: 整表聚合（长度 {len(replay_gen.data)}）+ "
      f"逐节点 { {k: len(g.data) for k, g in node_generators.items()} }")

# 供逐序列绘图/断言使用的宽表（index=date，columns=SYN_SKU_1..5）
wide_df = pivot_demand_wide(demand_df)
demand_df.head(3)

In [ ]:
# ---- 三种官方基线（Naive=历史均值 / MovingAverage(window=10) / ES(alpha=0.3)）----
# 长表输入自动 pivot；内部经框架继承方法 generate_forecasts 做滚动预测，
# 输出整段 (T,) 的 mean/std，与真实需求逐期对齐计算 MAE / RMSE / MAPE
#（MAPE 对真实值为 0 的期做掩码）。多序列按全部序列拼接汇总（等长时=逐列平均）。
metrics_df = run_baseline_comparison(demand_df)
print(f"基线误差对比 [{data_source}]：")
print(metrics_df.round(3))

In [ ]:
# ---- 自定义预测器：加权移动平均（近期权重更大）----
# mean = 最近 window 期加权平均；std = 历史滚动"1步加权预测 vs 真实值"残差标准差
#（数据驱动的预测不确定性，非写死常数）。等权 WMA 的 mean 与 MovingAverage 数学上
# 等价，因此这里用非均匀权重让对比有区分度。
wma = WeightedMAForecaster(window=8, weights=[1, 1, 1, 1, 2, 2, 3, 4])

# 加入基线对比流程：前 3 行官方基线 + 第 4 行 WeightedMA，同一数据同一口径
combined_df = run_comparison_with_custom(demand_df, forecaster=wma)
print(f"含自定义预测器的误差对比 [{data_source}]：")
print(combined_df.round(3))

# (mean, std) 数组契约演示：长度 == steps_ahead，std 供下游补货策略使用
col = wide_df.columns[0]                                  # 第一条序列（宽表列名）
hist = wide_df[col].to_numpy(dtype=np.float64)
fm8, fs8 = wma.forecast(hist, steps_ahead=8)
assert isinstance(fm8, np.ndarray) and isinstance(fs8, np.ndarray)
assert fm8.shape == (8,) and fs8.shape == (8,)
print(f"多步预测契约: mean{fm8.shape} / std{fs8.shape}；"
      f"末点 mean={fm8[-1]:.2f}, std={fs8[-1]:.2f}")

In [ ]:
# ---- 图1：真实需求 vs 三种基线 vs 自定义预测的均值曲线 ----
# 绘图逻辑沉淀在 src/visualization/dashboard.py（长表输入自动 pivot）。
col = wide_df.columns[0]                                 # 换列名可看其它序列
baseline_fc = generate_baseline_forecasts(wide_df[[col]])   # {序列: {预测器: (fm, fs)}}
custom_fm, custom_fs = wma.generate_forecasts(hist)         # 整段 (T,) 的 (mean, std)

forecast_map = dict(baseline_fc[col])                       # 三种基线
forecast_map["WeightedMA"] = (custom_fm, custom_fs)         # 自定义预测器红色高亮
fig1 = plot_forecast_mean(demand_df, col, forecast_map,     # 长表输入，自动 pivot
                          data_source=data_source,
                          save_path=FIGURES_DIR / "fig1_baseline_vs_custom_mean.png")
plt.show()
print(f"图1已保存: {FIGURES_DIR / 'fig1_baseline_vs_custom_mean.png'}")

In [ ]:
# ---- 图2：预测 std 不确定性（阴影填充）----
# 上图：WeightedMA 均值 ±1σ / ±2σ 区间（σ = 历史滚动预测残差标准差，随期数变化）；
# 下图：WeightedMA 与 ExponentialSmoothing 的 std(t) 曲线对比——直观展示不确定度
# 随历史的演化。绘图逻辑沉淀在 src/visualization/dashboard.py。
es_fs = baseline_fc[col]["ExponentialSmoothing"][1]

fig2 = plot_uncertainty_band(demand_df, col, custom_fm, custom_fs,
                             std_curves={"ExponentialSmoothing": es_fs},
                             data_source=data_source,
                             save_path=FIGURES_DIR / "fig2_forecast_uncertainty_band.png")
plt.show()
print(f"图2已保存: {FIGURES_DIR / 'fig2_forecast_uncertainty_band.png'}")

In [ ]:
# ---- 误差指标保存到 outputs/results/（UTF-8-SIG，Excel 直接打开不乱码）----
path_base = save_baseline_metrics(metrics_df)      # 3 官方基线 -> baseline_metrics.csv
path_comb = save_baseline_metrics(combined_df,     # 含 WeightedMA 的 4 行表
                                  "baseline_metrics_with_custom.csv")
print(f"已保存（当前口径: {data_source}）:\n  {path_base}\n  {path_comb}")

# 交接提示：这两个 CSV 是 custom_forecaster 后续迭代（换权重/加预测器）的对比基准；
# (fm, fs) 数组即交给仿真的预测交接物，按节点名注入 NetworkSupplyChain 的叶子节点。

## 小结
- **数据口径**：长表 `date/store_id/item_id/demand`（团队注意事项 §8）；合成默认
  参数对齐 config 基准（5 SKU × 365 天、demand.mean=10、demand.std=3，seed=42 可复现）。
- **误差排序**见 Cell 4 输出表（合成演示口径，重跑自动更新）。
- **交接物**：metrics CSV（outputs/results/）+ 每序列整段 `(fm, fs)` 数组
  ——`forecast()` 恒返回 `(mean, std)` 两个 `np.ndarray`，长度等于 `steps_ahead`；
  需求数据侧为 `ReplayDemandGenerator`（整表聚合 + 逐节点过滤两种构造）。
- **长表→宽表**：`pivot_demand_wide(demand_df)`（index=date，columns=序列名），
  仿真层按列名映射拓扑节点。